Initial Exploration & Data Integrity

In [2]:
import pandas as pd
import numpy as np

# 1. Load the dataset
df = pd.read_csv('/content/AB_NYC_2019.csv')

# 2. Check the dimensions and layout
print("--- Dataset Shape ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")

print("--- Column Summary & Data Types ---")
print(df.info())

print("\n--- Missing Value Counts ---")
print(df.isnull().sum())

--- Dataset Shape ---
Rows: 43427, Columns: 16

--- Column Summary & Data Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43427 entries, 0 to 43426
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              43427 non-null  int64  
 1   name                            43411 non-null  object 
 2   host_id                         43426 non-null  float64
 3   host_name                       43405 non-null  object 
 4   neighbourhood_group             43426 non-null  object 
 5   neighbourhood                   43426 non-null  object 
 6   latitude                        43426 non-null  float64
 7   longitude                       43426 non-null  float64
 8   room_type                       43426 non-null  object 
 9   price                           43426 non-null  float64
 10  minimum_nights                  43426 non-null  float64
 11  number_of

Strategic Missing Data Handling

In [3]:
# Create a copy to protect original data integrity
df_clean = df.copy()

# 1. Impute numerical missing values with 0
df_clean['reviews_per_month'] = df_clean['reviews_per_month'].fillna(0.0)

# 2. Fill missing text descriptions with placeholders
df_clean['name'] = df_clean['name'].fillna("Unknown Listing")
df_clean['host_name'] = df_clean['host_name'].fillna("Unknown Host")

# 3. Ensure last_review is treated as a date type
df_clean['last_review'] = pd.to_datetime(df_clean['last_review'], errors='coerce')

# Verify that missing values are resolved
print("Remaining missing values:")
print(df_clean[['name', 'host_name', 'reviews_per_month']].isnull().sum())

Remaining missing values:
name                 0
host_name            0
reviews_per_month    0
dtype: int64


Duplicate Records Audit

In [4]:
# Check for exact row duplicates
duplicate_count = df_clean.duplicated().sum()
print(f"Number of duplicate rows found: {duplicate_count}")

# Check for duplicate listing IDs (since 'id' should be completely unique)
id_duplicate_count = df_clean.duplicated(subset=['id']).sum()
print(f"Number of duplicate listing IDs found: {id_duplicate_count}")

# If duplicates exist, remove them keeping the first occurrence
if duplicate_count > 0 or id_duplicate_count > 0:
    df_clean = df_clean.drop_duplicates(subset=['id'], keep='first')
    print("Duplicates successfully removed!")

Number of duplicate rows found: 0
Number of duplicate listing IDs found: 0


Text Standardization & Data Type Consistency

In [8]:
# Explicitly filter out rows where critical columns have NaN values.
# This ensures that no NaNs remain in 'price', 'minimum_nights', 'neighbourhood_group', or 'room_type'.
initial_rows = len(df_clean)
df_clean = df_clean[df_clean['price'].notna() &
                    df_clean['minimum_nights'].notna() &
                    df_clean['neighbourhood_group'].notna() &
                    df_clean['room_type'].notna()]
rows_after_filter = len(df_clean)

if initial_rows - rows_after_filter > 0:
    print(f"Removed {initial_rows - rows_after_filter} rows with critical NaN values to ensure data integrity.")

# Now verify that no NaNs exist in these columns before conversion
print("\nNaNs in price after filtering:", df_clean['price'].isnull().sum())
print("NaNs in minimum_nights after filtering:", df_clean['minimum_nights'].isnull().sum())
print("NaNs in neighbourhood_group after filtering:", df_clean['neighbourhood_group'].isnull().sum())
print("NaNs in room_type after filtering:", df_clean['room_type'].isnull().sum())

# Strip accidental whitespace and ensure consistent string format
df_clean['neighbourhood_group'] = df_clean['neighbourhood_group'].str.strip()
df_clean['neighbourhood'] = df_clean['neighbourhood'].str.strip()
df_clean['room_type'] = df_clean['room_type'].str.strip()

# Verify structural unique values match perfectly
print("Unique Neighbourhood Groups:", df_clean['neighbourhood_group'].unique())
print("Unique Room Types:", df_clean['room_type'].unique())

# Ensure metric columns are explicit numerical data types
df_clean['price'] = df_clean['price'].astype(int)
df_clean['minimum_nights'] = df_clean['minimum_nights'].astype(int)


Removed 1 rows with critical NaN values to ensure data integrity.

NaNs in price after filtering: 0
NaNs in minimum_nights after filtering: 0
NaNs in neighbourhood_group after filtering: 0
NaNs in room_type after filtering: 0
Unique Neighbourhood Groups: ['Brooklyn' 'Manhattan' 'Queens' 'Staten Island' 'Bronx']
Unique Room Types: ['Private room' 'Entire home/apt' 'Shared room']


/tmp/ipykernel_1870/3434927372.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['neighbourhood_group'] = df_clean['neighbourhood_group'].str.strip()
/tmp/ipykernel_1870/3434927372.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['neighbourhood'] = df_clean['neighbourhood'].str.strip()
/tmp/ipykernel_1870/3434927372.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cavea

Outlier Detection and Trimming

In [9]:
# 1. Eliminate impossible pricing entries
df_clean = df_clean[df_clean['price'] > 0]

# 2. Calculate IQR bounds for pricing
Q1 = df_clean['price'].quantile(0.25)
Q3 = df_clean['price'].quantile(0.75)
IQR = Q3 - Q1
upper_limit_price = Q3 + 1.5 * IQR

print(f"Statistical upper bound threshold for price: ${upper_limit_price:.2f}")

# Filter price outliers (Alternatively, use an explicit logical limit like $1000)
df_clean = df_clean[df_clean['price'] <= upper_limit_price]

# 3. Handle extreme minimum nights outliers (e.g., filtering out entries requiring > 30 nights)
df_clean = df_clean[df_clean['minimum_nights'] <= 30]

print(f"Dataset shape after removing structural outliers: {df_clean.shape}")

Statistical upper bound threshold for price: $334.00
Dataset shape after removing structural outliers: (40408, 16)


Export the Cleaned Dataset

In [10]:
# Save out the polished dataset
output_path = '/content/AB_NYC_2019_Cleaned.csv'
df_clean.to_csv(output_path, index=False)

print(f"🎉 Success! Your cleaned dataset has been compiled and saved to: {output_path}")

🎉 Success! Your cleaned dataset has been compiled and saved to: /content/AB_NYC_2019_Cleaned.csv
